In [0]:
from pyspark.sql import functions as F

catalog = "severn_trent"
silver_schema = "silver"
gold_schema = "gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{gold_schema}")

# -----------------------------------------------------------
# Helper function to refresh Gold Delta tables
# -----------------------------------------------------------
def write_gold_table(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{catalog}.{gold_schema}.{table_name}")
    )

    print(f"Gold table created: {catalog}.{gold_schema}.{table_name}")


# -----------------------------------------------------------
# Gold Dimensions
# Keep only the latest/current SCD2 version of each dimension.
# -----------------------------------------------------------

dim_cancellation_reason = (
    spark.table(f"{catalog}.{silver_schema}.DimCancellationReason")
    .filter(F.col("CurrentFlag") == "Y")
    .select(
        "cancellation_id",
        "cancellation_reason",
        "reason_group",
        "is_customer_driven",
        "severity"
    )
)

dim_county = (
    spark.table(f"{catalog}.{silver_schema}.DimCounty")
    .filter(F.col("CurrentFlag") == "Y")
    .select(
        "county_id",
        "county_name",
        "region",
        "country",
        "population_band",
        "urban_rural"
    )
)

dim_people = (
    spark.table(f"{catalog}.{silver_schema}.DimPeople")
    .filter(F.col("CurrentFlag") == "Y")
    .select(
        "people_id",
        "first_name",
        "surname",
        "employee_name",
        "job_role",
        "employment_type",
        "team",
        "hire_date"
    )
)

dim_shrinkage_type = (
    spark.table(f"{catalog}.{silver_schema}.DimShrinkageType")
    .filter(F.col("CurrentFlag") == "Y")
    .select(
        "shrinkage_type_id",
        "shrinkage_type",
        "category",
        "is_paid",
        "is_planned"
    )
)

write_gold_table(dim_cancellation_reason, "DimCancellationReason")
write_gold_table(dim_county, "DimCounty")
write_gold_table(dim_people, "DimPeople")
write_gold_table(dim_shrinkage_type, "DimShrinkageType")


# -----------------------------------------------------------
# Gold Facts
# Silver fact tables containing the latest SCD1 state and added calculated hours column
# -----------------------------------------------------------

fact_shift = (
    spark.table(f"{catalog}.{silver_schema}.FactShift")
    .withColumn("scheduled_date", F.to_date("scheduled_start"))
    .withColumn(
        "shift_hours",
        (
            F.unix_timestamp("scheduled_end")
            - F.unix_timestamp("scheduled_start")
        ) / 3600
    )
)

fact_work = (
    spark.table(f"{catalog}.{silver_schema}.FactWork")
    .withColumn("scheduled_date", F.to_date("scheduled_start"))
    .withColumn(
        "work_hours",
        (
            F.unix_timestamp("scheduled_end")
            - F.unix_timestamp("scheduled_start")
        ) / 3600
    )
)

fact_shrinkage = (
    spark.table(f"{catalog}.{silver_schema}.FactShrinkage")
    .withColumn("shrinkage_date", F.to_date("start_datetime"))
    .withColumn(
        "shrinkage_hours",
        (
            F.unix_timestamp("end_datetime")
            - F.unix_timestamp("start_datetime")
        ) / 3600
    )
)

write_gold_table(fact_shift, "FactShift")
write_gold_table(fact_shrinkage, "FactShrinkage")
write_gold_table(fact_work, "FactWork")


# -----------------------------------------------------------
# Gold Date Dimension
# -----------------------------------------------------------

date_source_df = (
    fact_shift.select(F.col("scheduled_date").alias("calendar_date"))
    .unionByName(
        fact_shrinkage.select(F.col("shrinkage_date").alias("calendar_date"))
    )
    .unionByName(
        fact_work.select(F.col("scheduled_date").alias("calendar_date"))
    )
    .filter(F.col("calendar_date").isNotNull())
)

date_bounds = date_source_df.agg(
    F.min("calendar_date").alias("min_date"),
    F.max("calendar_date").alias("max_date")
).first()

if date_bounds["min_date"] is not None:
    dim_date = (
        spark.range(1)
        .select(
            F.explode(
                F.sequence(
                    F.lit(date_bounds["min_date"]),
                    F.lit(date_bounds["max_date"])
                )
            ).alias("calendar_date")
        )
        .withColumn("date_key", F.date_format("calendar_date", "yyyyMMdd").cast("int"))
        .withColumn("year", F.year("calendar_date"))
        .withColumn("quarter", F.quarter("calendar_date"))
        .withColumn("month_number", F.month("calendar_date"))
        .withColumn("month_name", F.date_format("calendar_date", "MMMM"))
        .withColumn("day_of_month", F.dayofmonth("calendar_date"))
        .withColumn("day_name", F.date_format("calendar_date", "EEEE"))
        .withColumn("week_of_year", F.weekofyear("calendar_date"))
    )

    write_gold_table(dim_date, "DimDate")
    

print("Gold layer processing completed.")